# P3 · Базовый EDA: tgbn-trade и tgbn-genre

Базовые статистики, устройство **времени** и **message** (вес ребра), распределение вершин (степени user/item, разреженность bipartite-матрицы).

Источник — `*_edgelist.csv` (темпоральный граф: `t, src=user-side, dst=item-side, w=weight=message`) из пакета `py-tgb`. Конвенции: Polars (lazy для большого genre), Plotly, текст на русском. Вывод каждой ячейки проверяем по ходу.

In [1]:
import polars as pl
import plotly.express as px
import numpy as np

DS = "/Users/aleksandrpanysev/miniconda3/envs/tgb/lib/python3.13/site-packages/tgb/datasets"
# edgelist: (t, src=user-side, dst=item-side, w=weight=message)
EDGE_COLS = {
    "tgbn-trade": ("year", "nation", "trading nation", "weight"),
    "tgbn-genre": ("ts", "user_id", "genre", "weight"),
}
DATASETS = ["tgbn-trade", "tgbn-genre"]

def edges_lazy(name):
    t, s, d, w = EDGE_COLS[name]
    return (pl.scan_csv(f"{DS}/{name.replace('-', '_')}/{name}_edgelist.csv")
              .select([pl.col(t).alias("t"), pl.col(s).alias("src"),
                       pl.col(d).alias("dst"), pl.col(w).cast(pl.Float64).alias("w")]))

for nm in DATASETS:
    n = edges_lazy(nm).select(pl.len()).collect().item()
    print(f"{nm}: {n:,} рёбер в edgelist")

tgbn-trade: 468,245 рёбер в edgelist


tgbn-genre: 17,858,395 рёбер в edgelist


## 1. Базовые статистики

Размер графа, число user/item-вершин и асимметрия `N_user / N_item`, временной диапазон.

In [2]:
def basic_stats(name):
    a = edges_lazy(name).select([
        pl.len().alias("рёбра"),
        pl.col("src").n_unique().alias("N_user"),
        pl.col("dst").n_unique().alias("N_item"),
        pl.col("t").n_unique().alias("распц_t"),
        pl.col("t").min().alias("t_min"), pl.col("t").max().alias("t_max"),
    ]).collect().to_dicts()[0]
    a = {"датасет": name, **a}
    a["user/item"] = round(a["N_user"] / a["N_item"], 1)
    return a

stats = pl.DataFrame([basic_stats(nm) for nm in DATASETS]).select(
    ["датасет", "рёбра", "N_user", "N_item", "user/item", "распц_t", "t_min", "t_max"])
print("распц_t = число различных временных меток в edgelist")
stats

распц_t = число различных временных меток в edgelist


датасет,рёбра,N_user,N_item,user/item,распц_t,t_min,t_max
str,i64,i64,i64,f64,i64,i64,i64
"""tgbn-trade""",468245,254,254,1.0,31,1986,2016
"""tgbn-genre""",17858395,992,513,1.9,4187046,1108357203,1245461220


## 2. Время

trade — годовые метки (дискретно, 31 шаг); genre — unix-секунды (почти непрерывно). Смотрим темпоральную плотность (рёбра во времени) и грануляцию.

In [4]:
# trade: рёбер по годам
trade_t = (edges_lazy("tgbn-trade").group_by("t").agg(pl.len().alias("рёбра"))
                                   .sort("t").collect())
px.bar(trade_t.to_pandas(), x="t", y="рёбра",
       title="tgbn-trade: рёбер по годам").show()

# genre: бинируем unix-время в 120 корзин, переводим в дату
lf = edges_lazy("tgbn-genre")
tmin, tmax = lf.select([pl.col("t").min().alias("a"), pl.col("t").max().alias("b")]).collect().row(0)
NB = 120
genre_t = (lf.with_columns((((pl.col("t") - tmin) / (tmax - tmin) * (NB - 1)).floor()).alias("bin"))
             .group_by("bin").agg(pl.len().alias("рёбра")).sort("bin").collect()
             .with_columns((tmin + (pl.col("bin") + 0.5) / NB * (tmax - tmin)).cast(pl.Int64).alias("ts"))
             .with_columns(pl.from_epoch("ts").alias("дата")))
px.line(genre_t.to_pandas(), x="дата", y="рёбра",
        title="tgbn-genre: рёбер во времени (120 корзин)").show()

print(f"trade: {trade_t.height} годовых меток, рёбер/год: "
      f"мин {trade_t['рёбра'].min()}, медиана {trade_t['рёбра'].median():.0f}, макс {trade_t['рёбра'].max()}")
print(f"genre: span ≈ {(tmax - tmin)/86400/365:.1f} лет, "
      f"~{17858395/((tmax-tmin)/86400):.0f} рёбер/день в среднем")

trade: 31 годовых меток, рёбер/год: мин 9330, медиана 16475, макс 19371
genre: span ≈ 4.3 лет, ~11254 рёбер/день в среднем


## 3. Message (вес ребра)

В обоих датасетах `msg` одномерный — это `weight` ребра (`data.msg.size(-1) == 1`). Смотрим распределение веса: масштаб, тяжесть хвоста, доля нулей.